# Claude Developer Platform — ハンズオン

**所要時間:** 50分 | **レベル:** 300（中級） | **モデル:** Claude Sonnet 4.6 (`claude-sonnet-4–6`)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

## 何を作るか

チケットの詳細を読み取り、ナレッジベースを検索し、構造化された解決策を生成するマルチツール対応の**サポートチケットエージェント**です。フレームワークを使わず、Claude APIを直接使用します。

**顧客シナリオ:** TechFlow（ミドルマーケットB2B SaaS、1日500件以上のチケット）は、Tier 1のトリアージを自動化して人間のエージェントが複雑なエスカレーションに集中できるようにしたいと考えています。

## 学習目標

1. マルチツールのエージェントループを実装する（`while stop_reason == "tool_use"`）
2. 単一エージェント内でツール呼び出しと構造化出力を組み合わせる
3. `output_config.effort` によるアダプティブ思考を統合して推論の深さを制御する
4. 思考、ツール呼び出し、応答をリアルタイムでストリーミングする

## このセッションの位置づけ

Claudeを使って構築する方法はいくつかあります。このセッションでは **Messages API** に焦点を当てます — リクエスト/レスポンスサイクルを完全に制御できる最低レベルのインターフェースです:

| サーフェス | 概要 | 使用場面 |
|---------|-----------|-------------|
| **Messages API** ← *このセッション* | ClaudeへのダイレクトなHTTP/SDK呼び出し | エージェントループの完全制御、カスタムオーケストレーション、プロダクションバックエンド |
| **Agent SDK** | ツールディスパッチ組み込みのPythonフレームワーク | エージェントの迅速なプロトタイピング、オーケストレーションをフレームワークに任せたい場合 |
| **Claude Code** | CLIベースのコーディングエージェント | 開発者の生産性向上、リポジトリレベルのタスク、インタラクティブなコーディング |
| **claude.ai / Claude for Enterprise** | Projects・MCP付きチャットインターフェース | エンドユーザーのワークフロー、エンタープライズの知識業務、非開発者のユースケース |

今日は生のループを構築します。これを理解することで、その上にある全てがより明確になります。

## セットアップ

以下のセルを実行して依存関係のインストール、APIキーの設定、モックデータの読み込みを行ってください。



In [ ]:
# ── インストール & インポート ──
%pip install -q anthropic --upgrade

import anthropic
import json
import time
import os
from IPython.display import display, Markdown

# ── APIキー設定 ──
os.environ["ANTHROPIC_API_KEY"] = ""  # <-- ここにAnthropic APIキーを貼り付けてください

client = anthropic.Anthropic(timeout=900.0)  # タイムアウトを延長: ストリーミング非使用時にmax_tokens>21333で必要
MODEL = "claude-sonnet-4-6"

# ── 事前チェック ──
errors = []
if not os.environ.get("ANTHROPIC_API_KEY"):
    errors.append("❌ ANTHROPIC_API_KEY が設定されていません。Colabシークレット（🔑サイドバー）を使用するか、上に貼り付けてください。")

sdk_version = anthropic.__version__
print(f"SDKバージョン: {sdk_version}")

if not errors:
    try:
        test = client.messages.create(
            model=MODEL, max_tokens=1024,
            messages=[{"role": "user", "content": "Reply with only: ready"}],
            thinking={"type": "adaptive"},
        )
        text = "".join(b.text for b in test.content if b.type == "text").strip()
        print(f"✅ モデル: {MODEL}")
        print(f"✅ API接続確認 — テストレスポンス: {text}")
    except anthropic.AuthenticationError:
        errors.append("❌ APIキーが無効です。キーを確認して再試行してください。")
    except anthropic.BadRequestError as e:
        errors.append(f"❌ APIエラー: {e}。SDKの更新が必要な場合があります: %pip install -q --upgrade anthropic")
    except Exception as e:
        errors.append(f"❌ 接続エラー: {e}")

if errors:
    print("\n⚠️  セットアップの問題が検出されました:")
    for err in errors:
        print(f"   {err}")
    print("\n上記の問題を修正してこのセルを再実行してください。")
else:
    print("\n🚀 構築を始めましょう!")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 469.4/469.4 kB 396.5 kB/s eta 0:00:00
✅ API key loaded from Colab Secrets
SDK version: 0.86.0
✅ Model: claude-sonnet-4-6
✅ API connected — test response: ready

🚀 Ready to build!


In [ ]:
# ── サンプルチケットデータ ──

TICKETS = {
    "TKT-1042": {
        "id": "TKT-1042", "customer": "Acme Corp", "priority": "high",
        "product_area": "billing",
        "description": "We were charged twice for our March invoice. Invoice #INV-2024-0342 shows $4,500 but our bank shows two identical charges on March 3rd. Need immediate refund of the duplicate charge.",
        "status": "open"
    },
    "TKT-1043": {
        "id": "TKT-1043", "customer": "DataFlow Inc", "priority": "medium",
        "product_area": "api",
        "description": "Our webhook endpoint stopped receiving events after we rotated API keys yesterday. We've verified the new key works for REST calls but webhooks are still failing. Getting 401 errors in the webhook logs.",
        "status": "open"
    },
    "TKT-1044": {
        "id": "TKT-1044", "customer": "CloudScale Ltd", "priority": "low",
        "product_area": "feature_request",
        "description": "Would love to see bulk export functionality in the dashboard. Currently we have to export reports one at a time which is painful when we need quarterly summaries across 50+ projects.",
        "status": "open"
    },
    "TKT-1045": {
        "id": "TKT-1045", "customer": "SecureNet Systems", "priority": "critical",
        "product_area": "account",
        "description": "Our admin account (admin@securenet.io) is locked out after failed MFA attempts. We have 47 team members who can't access the platform because SSO is tied to this admin account. This is blocking all work.",
        "status": "open"
    },
    "TKT-1046": {
        "id": "TKT-1046", "customer": "MedTech Solutions", "priority": "high",
        "product_area": "api",
        "description": "Our production integration started returning intermittent 500 errors around 2am last night. About 15% of API calls are failing. We haven't changed anything on our end. Errors seem random - sometimes the same request works on retry. Our team in Singapore is blocked and we need this resolved ASAP.",
        "status": "open"
    },
}

KB_ARTICLES = {
    "KB-001": {"title": "Processing Duplicate Payment Refunds", "content": "For duplicate charges: 1) Verify the duplicate in the billing system, 2) Issue refund through the payment processor (takes 3-5 business days), 3) Send confirmation email with refund reference number. Escalate if amount exceeds $10,000."},
    "KB-002": {"title": "Webhook Authentication After Key Rotation", "content": "When API keys are rotated, webhook signing secrets must also be updated. Go to Settings > Webhooks > Edit endpoint, and regenerate the signing secret. The old secret is invalidated immediately on key rotation. Common mistake: rotating the API key but not the webhook signing secret."},
    "KB-003": {"title": "Bulk Export Feature (Roadmap)", "content": "Bulk export is on the Q3 roadmap. Workaround: Use the REST API's /reports/export endpoint with date range parameters to programmatically export multiple reports. See API docs for batch export examples."},
    "KB-004": {"title": "Admin Account Lockout Recovery", "content": "For locked admin accounts: 1) Verify identity through the secondary email on file, 2) Reset MFA through the admin recovery flow at /admin/recover, 3) Temporary access can be granted through support-level override (requires manager approval). Critical: If SSO is blocked, enable the bypass login at /login/direct for affected users."},
    "KB-005": {"title": "API Rate Limiting Best Practices", "content": "Default rate limits: 100 requests/minute for standard plans, 1000/minute for enterprise. Use exponential backoff with jitter for retries. Monitor usage via the X-RateLimit headers in responses."},
    "KB-006": {"title": "Invoice Discrepancy Resolution", "content": "For billing discrepancies: Check the billing audit log for the account, compare with payment processor records, and verify no pending transactions. Contact finance team for adjustments over $5,000."},
    "KB-007": {"title": "Intermittent 500 Errors Troubleshooting", "content": "For intermittent server errors: 1) Check the status page for known outages, 2) Review rate limit headers - 429s can masquerade as 500s behind load balancers, 3) Check if errors correlate with payload size or specific endpoints, 4) Enable request ID logging and contact support with specific request IDs for investigation. If >10% error rate persists for >1 hour, escalate to engineering."},
}

def get_ticket(ticket_id: str) -> str:
    ticket = TICKETS.get(ticket_id)
    if ticket:
        return json.dumps(ticket)
    return json.dumps({"error": f"Ticket {ticket_id} not found"})

def search_kb(query: str) -> str:
    query_lower = query.lower()
    results = []
    for article_id, article in KB_ARTICLES.items():
        if any(word in article["title"].lower() or word in article["content"].lower()
               for word in query_lower.split() if len(word) > 2):
            results.append({"id": article_id, **article})
    if not results:
        results = [{"id": "KB-000", "title": "No matches found", "content": "No relevant articles found. Consider escalating to Tier 2 support."}]
    return json.dumps(results[:3])

def resolve_ticket(ticket_id: str, resolution: str, status: str = "resolved") -> str:
    ticket = TICKETS.get(ticket_id)
    if ticket:
        ticket["status"] = status
        ticket["resolution"] = resolution
        return json.dumps({"success": True, "ticket_id": ticket_id, "new_status": status})
    return json.dumps({"error": f"Ticket {ticket_id} not found"})

TOOL_FUNCTIONS = {"get_ticket": get_ticket, "search_kb": search_kb, "resolve_ticket": resolve_ticket}

def execute_tool(name: str, input_data: dict) -> str:
    func = TOOL_FUNCTIONS.get(name)
    if func:
        return func(**input_data)
    return json.dumps({"error": f"Unknown tool: {name}"})

print("モックツールとサンプルデータを読み込みました!")
print(f"   利用可能なチケット: {', '.join(TICKETS.keys())}")
print(f"   ナレッジベース記事数: {len(KB_ARTICLES)}")


Mock tools and sample data loaded!
   Available tickets: TKT-1042, TKT-1043, TKT-1044, TKT-1045, TKT-1046
   Knowledge base articles: 7


---
# パート1: マルチツールエージェントループ（15分）

TechFlowのTier 1サポートチームは現在、各チケットを手動で処理しています: 顧客を検索し、ナレッジベースを検索し、問題を分類し、解決策を下書きする。これが1日500件以上、1件あたり約8分かかります。彼らはClaudeにこのワークフローを自律的に行ってほしいと考えています。

そのワークフローを置き換えるエージェントを構築しましょう — 3つのツール、1つのループ、フレームワークなし。

> **主要コンセプト:** Claude Sonnet 4.6は *アダプティブ思考* をサポートしています — タスクの複雑さに基づいて自動的にどれだけ推論するかを決定します。`thinking={"type": "adaptive"}` を全てのAPI呼び出しで有効にします。

## パート2: ツールスキーマの定義

TechFlowのサポートエージェントは3つのシステムにアクセスする必要があります: チケッティングプラットフォーム（詳細の検索）、ナレッジベース（解決策の検索）、解決エンジン（チケットのクローズ）。これらのそれぞれが、Claudeに何が利用可能でどのように呼び出すかを伝えるツールスキーマになります。

`description` フィールドは重要です — Claudeが特定のステップでどのツールを使うかを決める根拠となります。曖昧な説明はツール選択の誤りにつながります。

> 📖 **参考資料:** [ツール使用ドキュメント](https://platform.claude.com/docs/en/agents-and-tools/tool-use/overview)

In [ ]:
# TODO: search_kb と resolve_ticket のツールスキーマを定義する
# 各ツールに必要なもの: name（名前）、description（説明）、input_schema（propertiesとrequiredを含む）

tools = [
    # ✅ 例: get_ticket は完成しています — 次の2つの参考に使用してください
    {
        "name": "get_ticket",
        "description": "Retrieve full details for a support ticket by its ID, including customer, priority, product area, and description.",
        "input_schema": {
            "type": "object",
            "properties": {
                "ticket_id": {"type": "string", "description": "The ticket ID, e.g. TKT-1042"}
            },
            "required": ["ticket_id"]
        }
    },

    # TODO: search_kb を記入する
    # ヒント: descriptionはClaudeがこのツールをいつ呼ぶかを決める根拠 — 具体的に書くこと
    {
        "name": "search_kb",
        "description": "",  # <-- ここを記入する
        "input_schema": {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": ""}  # <-- 説明を記入する
            },
            "required": ["query"]
        }
    },

    # TODO: resolve_ticket を記入する
    # ヒント: statusはenumにすること — 3つの解決状態とは何か？
    {
        "name": "resolve_ticket",
        "description": "",  # <-- ここを記入する
        "input_schema": {
            "type": "object",
            "properties": {
                "ticket_id": {"type": "string"},
                "resolution": {"type": "string"},
                "status": {"type": "string", "enum": []}  # <-- enum値を記入する
            },
            "required": ["ticket_id", "resolution", "status"]
        }
    }
]

print(f"定義されたツールスキーマ数: {len(tools)}: {[t['name'] for t in tools]}")


## パート3: エージェントループの構築

これがコアとなる自動化です: チケットが届いたら、Claudeはそれを検索し、解決策を探し、解決します — 人間の介入なしに。エージェントループがこれを可能にします: `while response.stop_reason == "tool_use"` でツール呼び出しを抽出し、実行し、結果を追加し、APIを再度呼び出す。Claudeがシーケンスを決定し、あなたのコードはオーケストレーションするだけです。

**重要:** `response.content` をそのまま返す — thinking ブロックと tool_use ブロックが含まれる場合があります。Claude Sonnet 4.6は全ての呼び出しで `thinking={"type": "adaptive"}` を使用します。

> 📖 **参考資料:** [エージェントのツール使用パターン](https://docs.anthropic.com/en/docs/build-with-claude/tool-use/agentic-tool-use)

In [ ]:
SYSTEM_PROMPT = """あなたはTechFlowのTier 1サポートエージェントです。TechFlowはミドルマーケット企業向けにプロジェクト管理とチームコラボレーションツールを提供するB2B SaaSプラットフォームです。

## あなたの役割
受信したサポートチケットを調査し、ナレッジベースでソリューションを見つけ、具体的で実行可能なガイダンスとともにチケットを解決します。

## プロセス
1. まず必ず チケットを検索して全体のコンテキストを把握する
2. 関連するソリューションと手順をナレッジベースで検索する
3. 具体的な次のステップを含む詳細な解決策でチケットを解決する

## ガイドライン
- 徹底的に: 問題が単純に見えても、解決前に必ずKBを検索する
- 具体的に: 解決策には正確な手順、リンク、タイムフレームを含める
- 必要に応じてエスカレーション: 確信が低い場合や特権アクセスが必要な問題はエスカレーションとしてマークする
- 正確に分類: billing（請求）、technical（技術）、account（アカウント）、feature_request（機能要望）

## エスカレーション基準
- $10,000を超える金融問題
- セキュリティ関連のアカウント侵害
- エンジニアリングの介入が必要な問題
- Enterprise SLA（1時間以内の応答）の顧客

## TechFlowの製品プラン
- Starter ($29/ユーザー/月): 基本的なプロジェクト管理、5GBストレージ、メールサポート、最大5プロジェクト、コミュニティフォーラム
- Professional ($79/ユーザー/月): 高度なアナリティクス、100GBストレージ、優先サポート、APIアクセス、無制限プロジェクト、カスタムフィールド、ガントチャート、タイムトラッキング
- Enterprise (カスタム価格): SSO/SAML、無制限ストレージ、専任CSM、カスタムインテグレーション、SLA保証、監査ログ、高度なセキュリティ、カスタムブランディング、優先APIレート制限

## 一般的な問題カテゴリとルーティング
- Billing（請求）: 請求書の相違、支払い失敗、プラン変更、返金依頼、サブスクリプションのキャンセル、日割り計算の質問
- Technical（技術）: APIエラー、インテグレーションの問題、Webhookの障害、パフォーマンスの問題、データエクスポートの問題、ブラウザの互換性
- Account（アカウント）: ログインの問題、MFAの問題、SSO設定、権限の変更、チーム管理、ユーザープロビジョニング
- Feature Requests（機能要望）: 製品フィードバック、ロードマップの問い合わせ、回避策の依頼、ベータアクセスのリクエスト

## 応答テンプレート
請求の問題を解決する際は必ず含める: トランザクションID、返金タイムライン、確認メールの詳細。
技術的な問題を解決する際は必ず含める: 再現手順、利用可能な場合の回避策、エスカレーション時のエンジニアリングチケット番号。
アカウントの問題を解決する際は必ず含める: 実施したセキュリティ確認手順と付与した一時アクセス。

## SLA要件
- Starter: 24時間応答時間、営業時間のみ
- Professional: 4時間応答時間、拡張時間帯（午前6時〜午後10時）
- Enterprise: 1時間応答時間、24時間365日サポート、専用Slackチャンネル

## トーン
プロフェッショナルで、共感的で、ソリューション志向であること。解決策に入る前に顧客の不満を認める。利用可能な場合は顧客名を使用する。関連するガイダンスには特定の製品プランを参照する。"""


# TODO: run_agent(user_message) を実装する
# 1. ユーザーメッセージを含むメッセージリストを作成する
# 2. client.messages.create() を以下のパラメータで呼び出す:
#    - model=MODEL, max_tokens=32000, system=SYSTEM_PROMPT, tools=tools
#    - thinking={"type": "adaptive"}
#    - messages=messages
# 3. response.stop_reason == "tool_use" の間ループ:
#    a. response.content をループして tool_use ブロックを探す
#    b. execute_tool(block.name, block.input) で各ツールを実行する
#    c. tool_use_id とコンテンツを含む tool_result 辞書を構築する
#    d. アシスタントの応答 + ツール結果をメッセージに追加する
#       （思考ブロックを含む全コンテンツブロックを返すこと！）
#    e. APIを再度呼び出す
# 4. 最終的な応答を返す

def run_agent(user_message: str):
    """サポートチケットエージェントを実行する。"""
    messages = [{"role": "user", "content": user_message}]

    response = client.messages.create(
        model=MODEL,
        max_tokens=32000,
        system=SYSTEM_PROMPT,
        tools=tools,
        thinking={"type": "adaptive"},
        messages=messages
    )

    # TODO: Claudeがまだツールを使いたい間ループする
    while response.stop_reason == ___:  # <-- "Claudeがツールを呼びたい" stop_reason は何か？

        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                result = execute_tool(block.name, block.input)
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": ___,  # <-- このresultをツール呼び出しにリンクする `block` のフィールドは何か？
                    "content": str(result)
                })

        # TODO: アシスタントターン + ツール結果をメッセージに追加する
        # 重要: 全コンテンツブロック（思考ブロックを含む）を返す — テキストのみではない
        messages.append({"role": "assistant", "content": ___})  # <-- ここに何を入れる？
        messages.append({"role": "user", "content": tool_results})

        # TODO: 更新されたメッセージでAPIを再度呼び出す
        response = client.messages.create(
            ___  # <-- 上と同じパラメータ
        )

    return response


# テスト実行!
# response = run_agent("Resolve ticket TKT-1042")
# for block in response.content:
#     if block.type == "text" and block.text.strip():
#         print(f"\n 最終応答:\n{block.text}")


## パート4: 構造化出力の追加

TechFlowのダウンストリームシステムは機械が読めるような解決策を必要としています — チケッティングプラットフォームはデータベースを更新し、分析ダッシュボードはカテゴリを追跡し、エスカレーションルーターは `escalation_needed` フラグを確認します。フリーテキストの応答では不十分です。

`output_config.format` はClaudeのテキスト応答をJSONスキーマに一致させるよう制約します。**重要:** formatの制約は *全て* のテキスト出力に適用されるため、最終API呼び出しにのみ追加します — ツールループ完了後です。ループ中、ClaudeはformatConstraintなしで通常通りツールを使用します。ツールが完了したら、`output_config.format` と `tool_choice={"type": "none"}` で1回追加呼び出しを行い、構造化されたJSON解決策を取得します。

**注意:** アダプティブ思考では、応答に `[thinking, text]` ブロックが含まれる場合があります。構造化JSONは *最後* のテキストブロックにあります。

> 📖 **参考資料:** [構造化出力 / JSONモード](https://docs.anthropic.com/en/docs/build-with-claude/structured-outputs)

In [ ]:
# ✅ RESOLUTION_SCHEMA は下に定義済みです — 読んでから run_agent_structured() を実装してください
RESOLUTION_SCHEMA = {
    "type": "json_schema",
    "schema": {
        "type": "object",
        "properties": {
            "diagnosis": {"type": "string", "description": "Root cause analysis of the issue"},
            "solution_steps": {"type": "array", "items": {"type": "string"}, "description": "Ordered steps to resolve"},
            "confidence": {"type": "string", "enum": ["high", "medium", "low"]},
            "escalation_needed": {"type": "boolean"},
            "category": {"type": "string", "enum": ["billing", "technical", "account", "feature_request"]}
        },
        "required": ["diagnosis", "solution_steps", "confidence", "escalation_needed", "category"],
        "additionalProperties": False
    }
}


def get_structured_result(response) -> dict:
    """応答の最後のテキストブロックから構造化JSONを抽出する。"""
    # アダプティブ思考では、コンテンツが [thinking, text] になる場合がある — JSONは最後のテキストブロックにある
    text_blocks = [b for b in response.content if b.type == "text" and b.text.strip()]
    if text_blocks:
        return json.loads(text_blocks[-1].text)
    return None


def run_agent_structured(user_message: str) -> dict:
    """構造化JSON出力でエージェントを実行する。"""
    # ステップ1: ツールループを実行する — run_agent() と同じ、output_config はここでは不要
    # （formatは全テキスト出力を制約するため、アクティブな状態ではツールが動作しない）
    messages = [{"role": "user", "content": user_message}]
    response = client.messages.create(
        model=MODEL, max_tokens=32000, system=SYSTEM_PROMPT,
        tools=tools, thinking={"type": "adaptive"}, messages=messages
    )
    while response.stop_reason == "tool_use":
        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                result = execute_tool(block.name, block.input)
                tool_results.append({"type": "tool_result", "tool_use_id": block.id, "content": str(result)})
        messages.append({"role": "assistant", "content": response.content})
        messages.append({"role": "user", "content": tool_results})
        response = client.messages.create(
            model=MODEL, max_tokens=32000, system=SYSTEM_PROMPT,
            tools=tools, thinking={"type": "adaptive"}, messages=messages
        )

    # ステップ2: 構造化出力を取得するための最終呼び出し
    messages.append({"role": "user", "content": "Provide your structured resolution as JSON."})
    final = client.messages.create(
        model=MODEL,
        max_tokens=8000,
        system=SYSTEM_PROMPT,
        output_config=___,   # <-- {"format": RESOLUTION_SCHEMA} を使ってRESOLUTION_SCHEMAをここに渡す
        tool_choice=___,     # <-- さらなるツール呼び出しを防ぐ: {"type": "none"}
        thinking={"type": "adaptive"},
        messages=messages
    )
    return get_structured_result(final)


# result = run_agent_structured("Resolve ticket TKT-1042")
# print(json.dumps(result, indent=2))


---
### ✅ チェックポイント1 — 構造化出力を持つ動作するサポートチケットエージェント

チケットごとに2〜3のツールを呼び出し、構造化JSONを返すエージェントができているはずです。

**確認:** `run_agent_structured("Resolve ticket TKT-1044")` → category: feature_request、APIの回避策を提案

---
# パート2: アダプティブ思考（15分）

基本的なエージェントは動作しますが、TechFlowのサポートリーダーから懸念が上がります: "一部のチケットは単純です — 重複請求、パスワードリセット。他は本当に曖昧です。難しいチケットには深い推論が欲しいですが、簡単なものを遅くしたくありません。そして、エージェントが難しいチケットを間違えた時、*なぜ* その判断をしたのかわかりません。"

アダプティブ思考はこの両方の問題を解決します。`output_config.effort` パラメータを使用すると、Claudeがどの程度深く推論するかを制御できます — 複雑または曖昧なチケットには "high"、単純なものには "low"。思考トレースにより意思決定プロセスの可視性が得られ、エージェントがなぜエスカレーションを選んだか、どのKB記事を最も重視したかを監査できます。

> 📖 **参考資料:** [アダプティブ思考](https://docs.anthropic.com/en/docs/build-with-claude/extended-thinking)

## セル5: エフォートレベルの思考制御を追加する

`output_config.effort` を追加してClaudeがどの程度深く推論するかを制御します。高エフォートでは、Claudeはツール呼び出し間に詳細な思考トレースを生成します — 何を検索するか、KB結果を評価する、エスカレーションするかどうかを決定する。低エフォートでは、単純なチケットの推論を簡潔に保ちます。エージェントの意思決定プロセスを見ることができるように思考ブロックを表示します。

In [ ]:
# TODO: エージェントにエフォートレベルの思考制御を追加する
# 1. run_agent をコピー — thinking={"type": "adaptive"} と
#    output_config={"effort": effort} でツールループを実行する（format は含めない — 最終呼び出しまで取っておく）
# 2. ループ内で思考ブロックを表示する: block.type == "thinking"
# 3. ツールループ終了後、最終呼び出しを以下で行う:
#    - output_config={"effort": effort, "format": RESOLUTION_SCHEMA}
#    - tool_choice={"type": "none"}
#    - "構造化された解決策をJSONで提供してください。" のようなユーザーメッセージを追加する
# 4. get_structured_result() を使って最終応答を解析する

def run_agent_thinking(user_message: str, effort: str = "high") -> dict:
    """エフォート制御されたアダプティブ思考でエージェントを実行する。"""
    pass  # ここに実装を記述する


## セル6: アダプティブ思考を実際に探索する

MedTech Solutions（TKT-1046）は断続的な500エラーを報告しています — APIコールの15%が失敗、自社側では変更なし、昨夜の午前2時に開始。レート制限か？サーバー側の障害か？ネットワークの問題か？KBには完璧なマッチがないでしょう。

ここで思考が本領を発揮します。高エフォートで曖昧なチケットを実行し、同じチケットで低エフォートと比較してください。推論の深さがどのように変わるかを観察し、自問してください: TechFlowの1日500件のチケットに対して、単純なチケットと複雑なチケットをどのように異なるエフォートレベルにルーティングしますか？

In [ ]:
# 高エフォートで曖昧なチケットを実行 — 思考トレースを観察する
print("=== TKT-1046: 断続的なAPIエラー（曘昧なケース） ===\n")
result = run_agent_thinking("Resolve ticket TKT-1046", effort="high")
print(f"\n解決策:")
print(json.dumps(result, indent=2))

# 比較: 同じチケット、低エフォート
print(f"\n\n{'='*50}")
print("同じチケット、低エフォート")
print(f"{'='*50}\n")

for effort in ["high", "low"]:
    start = time.time()
    result = run_agent_thinking("Resolve ticket TKT-1046", effort=effort)
    elapsed = time.time() - start
    print(f"\n[effort={effort}] 確信度: {result['confidence']} | ステップ数: {len(result['solution_steps'])} | エスカレーション: {result['escalation_needed']} | 時間: {elapsed:.1f}s")


---
### ✅ チェックポイント2 — 推論の可視性 + エフォート制御を持つエージェント

エフォートの比較では、推論の深さと品質における観察可能な違いが見られるはずです。

---
# パート3: ストリーミング（10分）

TechFlowのサポートエージェントはAIトリアージシステムを監視するリアルタイムダッシュボードを使用しています。チケットが届いたとき、彼らはエージェントが動作しているのを*見る*必要があります — どのチケットを検索しているか、何を探しているか、どう推論しているか。15秒のスピナーの後にテキストの壁が現れても信頼を築けません。

ストリーミングがこれを解決します。`create()` を `stream()` に置き換えると、トークンがリアルタイムで届きます: 思考トレースはエージェントが推論する中でフローし、ツール呼び出しは実行されると表示され、構造化された解決策は最後にストリーミングされます。

> 📖 **参考資料:** [ストリーミングメッセージ](https://docs.anthropic.com/en/docs/build-with-claude/streaming)

## セル7: ストリーミングエージェントループ

`client.messages.create()` を `client.messages.stream()` に置き換えて、TechFlowのダッシュボードがエージェントの動作をリアルタイムで表示できるようにします。主要なイベントタイプを処理します: `thinking_delta`（推論トークン）、`text_delta`（最終応答）、`input_json_delta`（ツール引数）。

ストリームが完了した後、完全な応答オブジェクトを取得してループを継続するために `stream.get_final_message()` を使用します。

In [ ]:
# TODO: ストリーミングエージェントループを構築する
# 1. create() をコンテキストマネージャーを使った stream() に置き換える (with ... as stream:)
#    ツールループ中は output_config={"effort": effort} を使用する（format制約なし）
# 2. ストリームイベントを反復処理し、以下を処理する:
#    - content_block_start: content_block.type を確認する（thinking/tool_use/text）
#    - content_block_delta: thinking_delta、text_delta、input_json_delta を処理する
# 3. ストリーミング後、完全な応答を取得するために stream.get_final_message() を使用する
# 4. stop_reason が tool_use の場合、ツールを実行してループを継続する
# 5. ツールループ終了後、最終ストリーミング呼び出しを以下で行う:
#    - output_config={"effort": effort, "format": RESOLUTION_SCHEMA}
#    - tool_choice={"type": "none"}
# 6. 最終JSONに get_structured_result() を使用する
# 注意: stream() に thinking={"type": "adaptive"} を渡すこと

def run_agent_streaming(user_message: str, effort: str = "high") -> dict:
    """ストリーミング出力でエージェントを実行する。"""
    pass  # ここに実装を記述する


## セル8: フルデモ

SecureNet Systemsがクリティカルなチケットを提出しました — 管理者アカウントがロックアウトされ、47人のチームメンバーがブロックされています。ストリーミングを使用してフルエージェントを実行し、4つの機能がリアルタイムで組み合わさるのを確認してください: エージェントループがツール呼び出しをオーケストレーションし、構造化出力が解決策の形式を保証し、アダプティブ思考がセキュリティへの影響を推論し、ストリーミングがそれら全てを発生と同時に表示します。

In [ ]:
print("フルエージェントデモ: TKT-1045（アカウントロックアウト）の解決")
print("ストリーミング + アダプティブ思考 + ツール + 構造化出力")
print("=" * 60)

start = time.time()
result = run_agent_streaming("Resolve ticket TKT-1045")
elapsed = time.time() - start

print(f"\n\n{'=' * 60}")
print(f"合計時間: {elapsed:.1f}s")
print(f"\n構造化された解決策:")
print(json.dumps(result, indent=2))


---
### ✅ チェックポイント3 — リアルタイムエージェント

1回のエージェント実行で4つの機能が組み合わさっています:
エージェントループ + 構造化出力 + アダプティブ思考 + ストリーミング。

---
# エクストラクレジット

1. **ツール選択の制御** — `tool_choice: {"type": "none"}` を使ってClaudeがツールの呼び出しをやめるよう強制する
2. **エフォートの最適化** — `effort="high"`、`effort="medium"`、`effort="low"` で全チケットを処理する。品質/速度の比較表を作成する。
3. **バッチ処理** — Batch APIを使って全チケットを一度に処理する（コスト50%削減）。

## 参考資料

- [Claude APIドキュメント](https://docs.anthropic.com/en/docs)
- [ツール使用ガイド](https://docs.anthropic.com/en/docs/build-with-claude/tool-use)
- [アダプティブ思考](https://docs.anthropic.com/en/docs/build-with-claude/extended-thinking)
- [構造化出力](https://docs.anthropic.com/en/docs/build-with-claude/structured-outputs)
- [ストリーミング](https://docs.anthropic.com/en/docs/build-with-claude/streaming)
